In [1]:
import pandas as pd
import glob
import os
import urllib.request

folder_path = "../data/Premier_League/Not_merged"
url_live = "https://www.football-data.co.uk/mmz4281/2526/E0.csv"
live_file_path = os.path.join(folder_path, "E0_25_26_LIVE.csv")
urllib.request.urlretrieve(url_live, live_file_path)
files = glob.glob(folder_path + "*.csv")

file_pattern = os.path.join(folder_path, "*.csv")
files = glob.glob(file_pattern)

print(f"Found {len(files)} files")

Found 26 files


*TODO: POBIERANIE NIE TYLKO 2026 ALE WSZYSTKICH DANYCH Z LAT OD JAKICHS 2000 DO NAJNOWSZYCH*


In [2]:
core_columns =[
    'Div', 'Date', 'HomeTeam', 'AwayTeam',
    'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR',
    'Referee',
    'HS', 'AS', 'HST', 'AST', 'HC', 'AC',
    'HF', 'AF', 'HY', 'AY', 'HR', 'AR'
]

all_matches = []

for file in files:
    df = pd.read_csv(file, encoding='utf-8', on_bad_lines='skip')

    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce', format='mixed')

    cols_to_keep = [col for col in core_columns if col in df.columns]
    df = df[cols_to_keep]
    all_matches.append(df)


In [3]:
master_db = pd.concat(all_matches, ignore_index=True)

In [4]:
master_db = master_db.dropna(subset=['HomeTeam', 'AwayTeam'])
master_db=master_db.sort_values(by='Date').reset_index(drop=True)

In [5]:
def get_season(date):
    if pd.isna(date):
        return "Unknown"
    year = date.year
    month = date.month
    if month >= 7:
        return f"{year}/{year+1}"
    else:
        return f"{year-1}/{year}"

master_db['Season'] = master_db['Date'].apply(get_season)

In [6]:
output_dir = "../data/Premier_League"
output_filename = os.path.join(output_dir, "PremierLeague_WszystkieSezony.csv")

master_db.to_csv(output_filename, index=False)

**DANE Z PREMIER LEAGUE ZOSTALY SCALONE, WSTEPNIE OBROBIONE I WRZUCONE DO JEDNEGO PLIKU**

In [7]:
df=master_db.copy()
df.isna().sum()

Div         0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HTHG        0
HTAG        0
HTR         0
Referee     0
HS          0
AS          0
HST         0
AST         0
HC          0
AC          0
HF          0
AF          0
HY          0
AY          0
HR          0
AR          0
Season      0
dtype: int64

In [8]:
df = master_db.copy()
ftr_mapping = {'H': 2, 'D': 1, 'A': 0}
df['Target'] = df['FTR'].map(ftr_mapping)

In [9]:
def get_home_points(ftr):
    if ftr == 'H': return 3
    elif ftr == 'D': return 1
    else: return 0

def get_away_points(ftr):
    if ftr == 'A': return 3
    elif ftr == 'D': return 1
    else: return 0

In [10]:
df['HomePoints'] = df['FTR'].apply(get_home_points)
df['AwayPoints'] = df['FTR'].apply(get_away_points)

print(df[['Date', 'HomeTeam', 'AwayTeam','FTR', 'HomePoints', 'AwayPoints']].head(10))

        Date    HomeTeam       AwayTeam FTR  HomePoints  AwayPoints
0 2000-08-19       Leeds        Everton   H           3           0
1 2000-08-19   Liverpool       Bradford   H           3           0
2 2000-08-19  Sunderland        Arsenal   H           3           0
3 2000-08-19    Charlton       Man City   H           3           0
4 2000-08-19     Chelsea       West Ham   H           3           0
5 2000-08-19    Coventry  Middlesbrough   A           0           3
6 2000-08-19       Derby    Southampton   D           1           1
7 2000-08-19   Tottenham        Ipswich   H           3           0
8 2000-08-19   Leicester    Aston Villa   D           1           1
9 2000-08-20  Man United      Newcastle   H           3           0


In [11]:
df=df.sort_values(by='Date').reset_index(drop=True)

#home stats
df['Home_Last5_HomePts']=df.groupby('HomeTeam')['HomePoints'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

#offensive home stats
df['Home_Last5_GF']=df.groupby('HomeTeam')['FTHG'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeShots']=df.groupby('HomeTeam')['HS'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeShotsOT']=df.groupby('HomeTeam')['HST'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_Corners']=df.groupby('HomeTeam')['HC'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeFouls']=df.groupby('HomeTeam')['HF'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeYellows']=df.groupby('HomeTeam')['HY'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeReds']=df.groupby('HomeTeam')['HR'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

#defensive home stats
df['Home_Last5_GA']=df.groupby('HomeTeam')['FTAG'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeShotsAgainst']=df.groupby('HomeTeam')['AS'].transform(lambda x:x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeShotsOTAgainst']=df.groupby('HomeTeam')['AST'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_CornersAgainst']=df.groupby('HomeTeam')['AC'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeFoulsAgainst']=df.groupby('HomeTeam')['AF'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeYellowsAgainst']=df.groupby('HomeTeam')['AY'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Home_Last5_HomeRedsAgainst']=df.groupby('HomeTeam')['AR'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

In [12]:
#away stats
df['Away_Last5_AwayPts']=df.groupby('AwayTeam')['AwayPoints'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

#away offensive stats
df['Away_Last5_GF']=df.groupby('AwayTeam')['FTAG'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayShots']=df.groupby('AwayTeam')['AS'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayShotsOT']=df.groupby('AwayTeam')['AST'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_Corners']=df.groupby('AwayTeam')['AC'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayFouls']=df.groupby('AwayTeam')['AF'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayYellows']=df.groupby('AwayTeam')['AY'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayReds']=df.groupby('AwayTeam')['AR'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
#away defensive stats
df['Away_Last5_GA']=df.groupby('AwayTeam')['FTHG'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayShotsAgainst']=df.groupby('AwayTeam')['HS'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayShotsOTAgainst']=df.groupby('AwayTeam')['HST'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_CornersAgainst']=df.groupby('AwayTeam')['HC'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayFoulsAgainst']=df.groupby('AwayTeam')['HF'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayYellowsAgainst']=df.groupby('AwayTeam')['HY'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
df['Away_Last5_AwayRedsAgainst']=df.groupby('AwayTeam')['HR'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

In [13]:
home_form=df[['Date','HomeTeam','HomePoints','FTHG','FTAG','HS','AS','HST','AST','HC','AC','HF','AF','HY','AY','HR','AR']].copy()
home_form.columns = ['Date', 'Team', 'Points', 'GoalsFor', 'GoalsAgainst', 'ShotsFor', 'ShotsAgainst', 'ShotsOTFor', 'ShotsOTAgainst', 'CornersFor', 'CornersAgainst', 'FoulsFor', 'FoulsAgainst', 'YellowsFor', 'YellowsAgainst', 'RedsFor', 'RedsAgainst']

away_form = df[['Date', 'AwayTeam', 'AwayPoints', 'FTAG', 'FTHG', 'AS', 'HS', 'AST', 'HST', 'AC', 'HC', 'AF', 'HF', 'AY', 'HY', 'AR', 'HR']].copy()
away_form.columns = ['Date', 'Team', 'Points', 'GoalsFor', 'GoalsAgainst', 'ShotsFor', 'ShotsAgainst', 'ShotsOTFor', 'ShotsOTAgainst', 'CornersFor', 'CornersAgainst', 'FoulsFor', 'FoulsAgainst', 'YellowsFor', 'YellowsAgainst', 'RedsFor', 'RedsAgainst']


In [14]:
team_form=pd.concat([home_form, away_form]).sort_values(by=['Team','Date']).reset_index(drop=True)
stat_columns = ['Points', 'GoalsFor', 'GoalsAgainst', 'ShotsFor', 'ShotsAgainst', 'ShotsOTFor', 'ShotsOTAgainst', 'CornersFor', 'CornersAgainst', 'FoulsFor', 'FoulsAgainst', 'YellowsFor', 'YellowsAgainst', 'RedsFor', 'RedsAgainst']
for col in stat_columns:
    team_form[f'Overall_Last5_{col}']=team_form.groupby('Team')[col].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())

cols_to_keep = ['Date', 'Team'] + [f'Overall_Last5_{col}' for col in stat_columns]

team_form=team_form[cols_to_keep]

In [15]:
df = df.merge(team_form, left_on=['Date','HomeTeam'], right_on=['Date','Team'], how='left')
df.rename(columns=lambda x: f"Home_{x}" if x in team_form.columns and x not in ['Date', 'Team'] else x, inplace=True)
df.drop('Team',axis=1,inplace=True)

df = df.merge(team_form, left_on=['Date','AwayTeam'], right_on=['Date','Team'], how='left')
df.rename(columns=lambda x: f"Away_{x}" if x in team_form.columns and x not in ['Date', 'Team'] else x, inplace=True)
df.drop('Team',axis=1,inplace=True)


In [16]:
kolumny_do_podgladu = [
    'Date', 'HomeTeam', 'AwayTeam',
    'Home_Last5_HomePts', 'Away_Last5_AwayPts',
    'Home_Overall_Last5_Points', 'Away_Overall_Last5_Points',
    'Home_Overall_Last5_ShotsOTFor',
    'Away_Overall_Last5_YellowsFor'
]
print(df[kolumny_do_podgladu].tail(10))

           Date     HomeTeam        AwayTeam  Home_Last5_HomePts  \
9691 2026-03-03      Everton         Burnley                 0.4   
9692 2026-03-03        Leeds      Sunderland                 1.4   
9693 2026-03-03  Bournemouth       Brentford                 1.6   
9694 2026-03-03       Wolves       Liverpool                 1.0   
9695 2026-03-04       Fulham        West Ham                 2.0   
9696 2026-03-04     Brighton         Arsenal                 1.6   
9697 2026-03-04     Man City   Nott'm Forest                 2.2   
9698 2026-03-04    Newcastle      Man United                 1.2   
9699 2026-03-04  Aston Villa         Chelsea                 1.4   
9700 2026-03-05    Tottenham  Crystal Palace                 0.4   

      Away_Last5_AwayPts  Home_Overall_Last5_Points  \
9691                 1.0                        1.4   
9692                 0.4                        1.0   
9693                 2.4                        1.8   
9694                 1.6       

In [18]:
df = df.sort_values(by='Date').reset_index(drop=True)
df['H2H_Home_Pts']=df.groupby(['HomeTeam', 'AwayTeam'])['HomePoints'].transform(lambda x: x.shift(1).rolling(3,min_periods=1).mean())
df['H2H_Home_GF']=df.groupby(['HomeTeam', 'AwayTeam'])['FTHG'].transform(lambda x: x.shift(1).rolling(3,min_periods=1).mean())
df['H2H_Home_GA']=df.groupby(['HomeTeam', 'AwayTeam'])['FTAG'].transform(lambda x: x.shift(1).rolling(3,min_periods=1).mean())



In [19]:
print(df[df['HomeTeam'] == 'Liverpool'][['Date', 'HomeTeam', 'AwayTeam', 'H2H_Home_Pts', 'H2H_Home_GF', 'H2H_Home_GA']].tail(10))

           Date   HomeTeam       AwayTeam  H2H_Home_Pts  H2H_Home_GF  \
9506 2025-11-01  Liverpool    Aston Villa      2.333333     2.000000   
9521 2025-11-22  Liverpool  Nott'm Forest      2.000000     2.000000   
9548 2025-12-03  Liverpool     Sunderland      1.666667     1.333333   
9561 2025-12-13  Liverpool       Brighton      2.333333     2.333333   
9581 2025-12-27  Liverpool         Wolves      3.000000     2.000000   
9597 2026-01-01  Liverpool          Leeds      2.000000     3.666667   
9623 2026-01-17  Liverpool        Burnley      2.000000     1.666667   
9644 2026-01-31  Liverpool      Newcastle      3.000000     2.666667   
9659 2026-02-08  Liverpool       Man City      2.333333     1.333333   
9684 2026-02-28  Liverpool       West Ham      3.000000     2.000000   

      H2H_Home_GA  
9506     0.333333  
9521     1.000000  
9548     0.666667  
9561     1.666667  
9581     0.333333  
9597     1.666667  
9623     0.666667  
9644     1.000000  
9659     0.333333  
9684   

In [20]:
df['Date']=pd.to_datetime(df['Date'])
df=df.sort_values(by=['Date']).reset_index(drop=True)

home_dates=df[['Date', 'HomeTeam']].copy()
home_dates.columns = ['Date','Team']

away_dates=df[['Date','AwayTeam']].copy()
away_dates.columns = ['Date','Team']

all_matches = pd.concat([home_dates, away_dates]).sort_values(by=['Team','Date']).reset_index(drop=True)

In [21]:
all_matches['DaysOfRest']=all_matches.groupby('Team')['Date'].diff().dt.days
df = df.merge(all_matches, left_on=['Date','HomeTeam'], right_on=['Date','Team'], how='left')
df.rename(columns={'DaysOfRest':'Home_DaysOfRest'}, inplace=True)
df.drop('Team',axis=1,inplace=True)


df = df.merge(all_matches, left_on=['Date','AwayTeam'], right_on=['Date','Team'], how='left')
df.rename(columns={'DaysOfRest':'Away_DaysOfRest'}, inplace=True)
df.drop('Team',axis=1,inplace=True)

In [24]:
print(df[['Date','HomeTeam','AwayTeam','Home_DaysOfRest','Away_DaysOfRest']].tail(10))

           Date     HomeTeam        AwayTeam  Home_DaysOfRest  Away_DaysOfRest
9691 2026-03-03      Everton         Burnley              3.0              3.0
9692 2026-03-03        Leeds      Sunderland              3.0              3.0
9693 2026-03-03  Bournemouth       Brentford              3.0              3.0
9694 2026-03-03       Wolves       Liverpool              4.0              3.0
9695 2026-03-04    Newcastle      Man United              4.0              3.0
9696 2026-03-04     Brighton         Arsenal              3.0              3.0
9697 2026-03-04     Man City   Nott'm Forest              4.0              3.0
9698 2026-03-04       Fulham        West Ham              3.0              4.0
9699 2026-03-04  Aston Villa         Chelsea              5.0              3.0
9700 2026-03-05    Tottenham  Crystal Palace              4.0              4.0
